In [2]:
from nba_api.stats.endpoints import LeagueDashLineups
from nba_api.stats.endpoints import TeamPlayerOnOffDetails
from nba_api.stats.endpoints import LeagueDashPlayerStats
from nba_api.stats.endpoints import SynergyPlayTypes
from nba_api.stats.static import teams

import os
import numpy as np
import time
import random
import pandas as pd
from requests.exceptions import ConnectionError, Timeout

In [3]:
FOUR_FACTORS = [
    "EFG_PCT",
    "FTA_RATE",
    "TM_TOV_PCT",
    "OREB_PCT",
    "OPP_EFG_PCT",
    "OPP_FTA_RATE",
    "OPP_TOV_PCT",
    "OPP_OREB_PCT"
]

### Data Gathering

In [4]:
def get_lineups(season, cache_dir="cache"):

    os.makedirs(cache_dir, exist_ok=True)

    path = f"{cache_dir}/lineups_{season}.csv"

    # load cache
    if os.path.exists(path):
        print("Loading cached:", season)
        return pd.read_csv(path)


    retries = 8

    for attempt in range(retries):

        try:

            print(
                f"Downloading {season}, attempt {attempt+1}/{retries}"
            )

            df = LeagueDashLineups(
                season=season,
                season_type_all_star="Regular Season",
                measure_type_detailed_defense="Four Factors",
                per_mode_detailed="Per100Possessions",
                group_quantity=5,
                timeout=60
            ).get_data_frames()[0]


            df.to_csv(path, index=False)

            print("Saved", season)

            time.sleep(random.uniform(2,4))

            return df


        except (ConnectionError, Timeout) as e:

            wait = 10 * (attempt + 1)

            print(
                f"Connection failed. Waiting {wait}s..."
            )

            time.sleep(wait)


    raise Exception(
        f"Could not download lineup data for {season}"
    )

In [5]:
lineups = get_lineups("2021-22")

print(lineups.shape)
print(lineups.head())

Loading cached: 2021-22
(2000, 32)
  GROUP_SET                                  GROUP_ID  \
0   Lineups     -201145-203115-203932-203999-1628420-   
1   Lineups   -101108-203109-1626164-1628969-1629028-   
2   Lineups    -201144-202711-203497-1626220-1628378-   
3   Lineups   -201143-203935-1627759-1628369-1629057-   
4   Lineups  -201976-1626156-1626157-1629020-1630162-   

                                          GROUP_NAME     TEAM_ID  \
0  J. Green - W. Barton - A. Gordon - N. Jokic - ...  1610612743   
1  C. Paul - J. Crowder - D. Booker - M. Bridges ...  1610612756   
2  M. Conley - B. Bogdanovic - R. Gobert - R. O'N...  1610612762   
3  A. Horford - M. Smart - J. Brown - J. Tatum - ...  1610612738   
4  P. Beverley - D. Russell - K. Towns - J. Vande...  1610612750   

  TEAM_ABBREVIATION  GP   W   L  W_PCT    MIN  ...  MIN_RANK  EFG_PCT_RANK  \
0               DEN  41  26  15  0.634  761.0  ...         1          5593   
1               PHX  38  31   7  0.816  754.0  ...       

In [6]:
def parse_player_ids(group_id):

    return [
        int(x)
        for x in group_id.split("-")
        if x
    ]

In [7]:
def weighted_average(df):

    if len(df) == 0:
        return pd.Series(
            {x: np.nan for x in FOUR_FACTORS}
        )

    return pd.Series({
        col: np.average(
            df[col],
            weights=df["MIN"]
        )
        for col in FOUR_FACTORS
    })

In [8]:
def build_player_four_factors(lineups):

    rows = []

    for team_id, team_df in lineups.groupby("TEAM_ID"):

        players = set(
            p
            for lineup in team_df["PLAYER_IDS"]
            for p in lineup
        )

        for player_id in players:

            on = team_df[
                team_df["PLAYER_IDS"]
                .apply(lambda x: player_id in x)
            ]

            off = team_df[
                team_df["PLAYER_IDS"]
                .apply(lambda x: player_id not in x)
            ]

            # skip players with no off court minutes
            if len(on) == 0 or len(off) == 0:
                continue

            on_stats = weighted_average(on)
            off_stats = weighted_average(off)

            row = {
                "PLAYER_ID": player_id,
                "TEAM_ID": team_id
            }

            for stat in FOUR_FACTORS:

                row[f"{stat}_ON"] = on_stats[stat]
                row[f"{stat}_OFF"] = off_stats[stat]

                row[f"{stat}_DIFF"] = (
                    on_stats[stat]
                    -
                    off_stats[stat]
                )
            
            row["MIN_ON"] = on["MIN"].sum()
            row["MIN_OFF"] = off["MIN"].sum()

            rows.append(row)

    return pd.DataFrame(rows)

In [9]:
def get_player_stats(season, cache_dir="cache"):

    os.makedirs(cache_dir, exist_ok=True)

    path = f"{cache_dir}/player_stats_{season}.csv"

    # load cache
    if os.path.exists(path):
        print("Loading cached:", season)
        return pd.read_csv(path)


    retries = 8

    for attempt in range(retries):

        try:

            print(
                f"Downloading {season}, attempt {attempt+1}/{retries}"
            )

            df = LeagueDashPlayerStats(
                    season=season,
                    season_type_all_star="Regular Season",
                    measure_type_detailed_defense="Usage",
                    per_mode_detailed="Per100Possessions",
                    timeout=60
                ).get_data_frames()[0]


            df.to_csv(path, index=False)

            print("Saved", season)

            time.sleep(random.uniform(2,4))

            return df


        except (ConnectionError, Timeout) as e:

            wait = 10 * (attempt + 1)

            print(
                f"Connection failed. Waiting {wait}s..."
            )

            time.sleep(wait)


    raise Exception(
        f"Could not download lineup data for {season}"
    )

    

In [10]:
def get_play_types(season, cache_dir="cache"):
    PLAY_TYPES = [
        "Transition",
        "Isolation",
        "PRBallHandler",
        "PRRollman",
        "Postup",
        "Spotup",
        "Handoff",
        "Cut",
        "OffScreen",
        "OffRebound",
        "Misc"
    ]

    os.makedirs(cache_dir, exist_ok=True)
    path = f"{cache_dir}/play_types_{season}.csv"

    if os.path.exists(path):
        print("Loading cached:", season)
        return pd.read_csv(path)

    retries = 8
    dfs = []

    for play_type in PLAY_TYPES:

        success = False

        for attempt in range(retries):

            try:
                print(f"{play_type}: attempt {attempt+1}/{retries}")

                df = SynergyPlayTypes(
                    season=season,
                    season_type_all_star="Regular Season",
                    player_or_team_abbreviation="P",
                    play_type_nullable=play_type,
                    type_grouping_nullable="offensive",
                    per_mode_simple="Totals",
                    timeout=60
                ).get_data_frames()[0]

                print(play_type, df.shape)

                df["PLAY_TYPE"] = play_type
                dfs.append(df)

                success = True

                time.sleep(random.uniform(2, 4))
                break

            except (ConnectionError, Timeout) as e:
                wait = 10 * (attempt + 1)
                print(f"{play_type}: connection failed ({e}). Waiting {wait}s...")
                time.sleep(wait)

        if not success:
            raise RuntimeError(
                f"Failed to download {play_type} for {season} after {retries} attempts."
            )

    result = pd.concat(dfs, ignore_index=True)
    result.to_csv(path, index=False)

    print(f"Saved {season}")

    return result

In [11]:
def aggregate_play_types(play_types):
    play_types_features = (
        play_types
        .pivot(
            index="PLAYER_ID",
            columns="PLAY_TYPE",
            values="POSS_PCT"
        )
        .reset_index()
    )

    play_types_features["CREATION_PCT"] = (
        play_types_features["Isolation"]
        + play_types_features["PRBallHandler"]
        + play_types_features["Handoff"]
    )

    play_types_features["FINISHING_PCT"] = (
        play_types_features["PRRollman"]
        + play_types_features["Cut"]
        + play_types_features["OffRebound"]
    )

    play_types_features["SHOOTING_PCT"] = (
        play_types_features["Spotup"]
        + play_types_features["OffScreen"]
    )

    play_types_features["TRANSITION_PCT"] = (
        play_types_features["Transition"]
    )

    play_types_features["POSTUP_PCT"] = (
        play_types_features["Postup"]
    )

    play_types_features["MISC_PCT"] = (
        play_types_features["Misc"]
    )

    play_types_features = play_types_features[
        [
            "PLAYER_ID",
            "CREATION_PCT",
            "FINISHING_PCT",
            "SHOOTING_PCT",
            "TRANSITION_PCT",
            "POSTUP_PCT",
            "MISC_PCT",
        ]
    ]

    return play_types_features

In [12]:
seasons = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26"
]

all_features = []

for season in seasons:

    print("Starting", season)

    lineups = get_lineups(season)

    lineups["PLAYER_IDS"] = lineups["GROUP_ID"].apply(parse_player_ids)

    four_factors = build_player_four_factors(lineups)
    four_factors["SEASON"] = season

    player_stats = get_player_stats(season)

    play_types = get_play_types(season)

    play_types = (
        play_types
        .groupby(["PLAYER_ID", "PLAY_TYPE"])
        .apply(
            lambda g: pd.Series({
                "POSS_PCT": np.average(g["POSS_PCT"], weights=g["POSS"])
            }),
            include_groups=False
        )
        .reset_index()
    )

    play_types_features = aggregate_play_types(play_types)

    # Keep only the columns you want
    player_stats = player_stats[
        [
            "PLAYER_ID",
            "USG_PCT",
            "PCT_AST",
            "PCT_DREB",
            "PCT_STL",
            "PCT_BLK",
            "PCT_BLKA"
        ]
    ]

    # Merge onto the four-factor features
    four_factors = four_factors.merge(
        player_stats,
        on="PLAYER_ID",
        how="left"
    )

    four_factors = four_factors.merge(
            play_types_features,
            on="PLAYER_ID",
            how="left"
        )

    all_features.append(four_factors)

    time.sleep(2)

four_factor_df = pd.concat(
    all_features,
    ignore_index=True
)

Starting 2021-22
Loading cached: 2021-22
Loading cached: 2021-22
Loading cached: 2021-22
Starting 2022-23
Loading cached: 2022-23
Loading cached: 2022-23
Loading cached: 2022-23
Starting 2023-24
Loading cached: 2023-24
Loading cached: 2023-24
Loading cached: 2023-24
Starting 2024-25
Loading cached: 2024-25
Loading cached: 2024-25
Loading cached: 2024-25
Starting 2025-26
Loading cached: 2025-26
Loading cached: 2025-26
Loading cached: 2025-26


In [13]:
four_factor_df = four_factor_df[
    four_factor_df["MIN_ON"] + four_factor_df["MIN_OFF"] > 300
]

In [14]:
four_factor_df = four_factor_df[
    four_factor_df["MIN_ON"] > 500
]

In [15]:
four_factor_df.columns

Index(['PLAYER_ID', 'TEAM_ID', 'EFG_PCT_ON', 'EFG_PCT_OFF', 'EFG_PCT_DIFF',
       'FTA_RATE_ON', 'FTA_RATE_OFF', 'FTA_RATE_DIFF', 'TM_TOV_PCT_ON',
       'TM_TOV_PCT_OFF', 'TM_TOV_PCT_DIFF', 'OREB_PCT_ON', 'OREB_PCT_OFF',
       'OREB_PCT_DIFF', 'OPP_EFG_PCT_ON', 'OPP_EFG_PCT_OFF',
       'OPP_EFG_PCT_DIFF', 'OPP_FTA_RATE_ON', 'OPP_FTA_RATE_OFF',
       'OPP_FTA_RATE_DIFF', 'OPP_TOV_PCT_ON', 'OPP_TOV_PCT_OFF',
       'OPP_TOV_PCT_DIFF', 'OPP_OREB_PCT_ON', 'OPP_OREB_PCT_OFF',
       'OPP_OREB_PCT_DIFF', 'MIN_ON', 'MIN_OFF', 'SEASON', 'USG_PCT',
       'PCT_AST', 'PCT_DREB', 'PCT_STL', 'PCT_BLK', 'PCT_BLKA', 'CREATION_PCT',
       'FINISHING_PCT', 'SHOOTING_PCT', 'TRANSITION_PCT', 'POSTUP_PCT',
       'MISC_PCT'],
      dtype='object')

In [16]:
from nba_api.stats.endpoints import TeamPlayerOnOffDetails


def get_on_off_target(season):

    rows = []

    for team in teams.get_teams():

        print(team["full_name"])

        data = TeamPlayerOnOffDetails(
            team_id=team["id"],
            season=season,
            measure_type_detailed_defense="Advanced",
            per_mode_detailed="Per100Possessions"
        )

        on = data.get_data_frames()[1]
        off = data.get_data_frames()[2]


        on = on[[
            "VS_PLAYER_ID",
            "VS_PLAYER_NAME",
            "NET_RATING"
        ]]

        off = off[[
            "VS_PLAYER_ID",
            "NET_RATING"
        ]]


        df = on.merge(
            off,
            on="VS_PLAYER_ID",
            suffixes=("_ON", "_OFF")
        )


        df["TEAM_ID"] = team["id"]

        df["ON_OFF_NET"] = (
            df["NET_RATING_ON"]
            -
            df["NET_RATING_OFF"]
        )

        rows.append(df)


        time.sleep(1)


    return pd.concat(rows, ignore_index=True)

In [17]:
target_rows = []

for season in seasons:

    target = get_on_off_target(season)

    target["SEASON"] = season

    target_rows.append(target)


on_off_target = pd.concat(
    target_rows,
    ignore_index=True
)

Atlanta Hawks
Boston Celtics
Cleveland Cavaliers
New Orleans Pelicans
Chicago Bulls
Dallas Mavericks
Denver Nuggets
Golden State Warriors
Houston Rockets
Los Angeles Clippers
Los Angeles Lakers
Miami Heat
Milwaukee Bucks
Minnesota Timberwolves
Brooklyn Nets
New York Knicks
Orlando Magic
Indiana Pacers
Philadelphia 76ers
Phoenix Suns
Portland Trail Blazers
Sacramento Kings
San Antonio Spurs
Oklahoma City Thunder
Toronto Raptors
Utah Jazz
Memphis Grizzlies
Washington Wizards
Detroit Pistons
Charlotte Hornets
Atlanta Hawks
Boston Celtics
Cleveland Cavaliers
New Orleans Pelicans
Chicago Bulls
Dallas Mavericks
Denver Nuggets
Golden State Warriors
Houston Rockets
Los Angeles Clippers
Los Angeles Lakers
Miami Heat
Milwaukee Bucks
Minnesota Timberwolves
Brooklyn Nets
New York Knicks
Orlando Magic
Indiana Pacers
Philadelphia 76ers
Phoenix Suns
Portland Trail Blazers
Sacramento Kings
San Antonio Spurs
Oklahoma City Thunder
Toronto Raptors
Utah Jazz
Memphis Grizzlies
Washington Wizards
Detroit Pi

In [18]:
on_off_target

,VS_PLAYER_ID,VS_PLAYER_NAME,NET_RATING_ON,NET_RATING_OFF,TEAM_ID,ON_OFF_NET,SEASON
0,1627760,"Barber, Cat",34.6,-11.5,1610612737,46.1,2021-22
1,203992,"Bogdanović, Bogdan",4.5,-1.4,1610612737,5.9,2021-22
2,1630602,"Brown Jr., Chaundee",-10.9,3.1,1610612737,-14.0,2021-22
3,203991,"Capela, Clint",3.3,-0.3,1610612737,3.6,2021-22
4,1628381,"Collins, John",0.8,1.8,1610612737,-1.0,2021-22
...,...,...,...,...,...,...,...
3291,1629012,"Sexton, Collin",-0.5,4.1,1610612766,-4.6,2025-26
3292,1642354,"Simpson, KJ",-6.9,3.1,1610612766,-10.0,2025-26
3293,1630214,"Tillman, Xavier",4.2,10.5,1610612766,-6.3,2025-26
3294,1629632,"White, Coby",10.9,9.8,1610612766,1.1,2025-26


In [19]:
on_off_target = on_off_target.rename(
    columns={
        "VS_PLAYER_ID": "PLAYER_ID",
        "VS_PLAYER_NAME": "PLAYER_NAME"
    }
)

on_off_target = on_off_target.groupby(
    ["PLAYER_ID","SEASON"],
    as_index=False
).agg({
    "ON_OFF_NET":"mean"
})

In [20]:
four_factor_df

,PLAYER_ID,TEAM_ID,EFG_PCT_ON,EFG_PCT_OFF,EFG_PCT_DIFF,FTA_RATE_ON,FTA_RATE_OFF,FTA_RATE_DIFF,TM_TOV_PCT_ON,TM_TOV_PCT_OFF,...,PCT_DREB,PCT_STL,PCT_BLK,PCT_BLKA,CREATION_PCT,FINISHING_PCT,SHOOTING_PCT,TRANSITION_PCT,POSTUP_PCT,MISC_PCT
0,201568,1610612737,0.564719,0.540972,0.023746,0.258369,0.227632,0.030737,0.099157,0.118972,...,0.239,0.113,0.090,0.089,0.179,0.145,0.393,0.075,0.170,0.038
1,1629027,1610612737,0.557585,0.530894,0.026691,0.247377,0.220489,0.026888,0.110551,0.109120,...,0.126,0.183,0.030,0.369,0.712,NaN,0.081,0.135,NaN,0.055
5,1626153,1610612737,0.537530,0.557367,-0.019837,0.254525,0.235553,0.018972,0.101418,0.113958,...,0.171,0.381,0.141,0.104,0.371,NaN,NaN,0.175,NaN,0.090
7,1628989,1610612737,0.555205,0.545059,0.010146,0.225525,0.267974,-0.042449,0.110279,0.110126,...,0.141,0.176,0.138,0.109,0.317,0.069,0.399,0.178,NaN,0.034
8,1630168,1610612737,0.568463,0.546669,0.021794,0.272792,0.232310,0.040482,0.107845,0.110892,...,0.246,0.191,0.645,0.239,NaN,0.763,NaN,0.070,0.060,0.092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2715,1628970,1610612766,0.575969,0.567814,0.008156,0.222049,0.251620,-0.029571,0.136629,0.169823,...,0.228,0.136,0.152,0.176,0.275,0.142,0.301,0.171,0.070,0.042
2716,1641706,1610612766,0.583567,0.559576,0.023991,0.234697,0.217640,0.017057,0.145324,0.140422,...,0.180,0.224,0.246,0.168,0.427,0.106,0.254,0.152,0.019,0.042
2720,1631217,1610612766,0.581930,0.564193,0.017737,0.232869,0.221802,0.011067,0.136724,0.152309,...,0.285,0.197,0.465,0.226,NaN,0.664,NaN,0.138,0.046,0.108
2722,1630163,1610612766,0.586782,0.553482,0.033301,0.208694,0.260479,-0.051785,0.132696,0.161329,...,0.204,0.290,0.092,0.291,0.582,NaN,0.218,0.123,NaN,0.037


In [21]:
on_off_target.columns

Index(['PLAYER_ID', 'SEASON', 'ON_OFF_NET'], dtype='object')

In [22]:
model_df = four_factor_df.merge(
    on_off_target,
    on=[
        "PLAYER_ID",
        "SEASON"
    ],
    how="inner"
)

In [23]:
model_df.columns

Index(['PLAYER_ID', 'TEAM_ID', 'EFG_PCT_ON', 'EFG_PCT_OFF', 'EFG_PCT_DIFF',
       'FTA_RATE_ON', 'FTA_RATE_OFF', 'FTA_RATE_DIFF', 'TM_TOV_PCT_ON',
       'TM_TOV_PCT_OFF', 'TM_TOV_PCT_DIFF', 'OREB_PCT_ON', 'OREB_PCT_OFF',
       'OREB_PCT_DIFF', 'OPP_EFG_PCT_ON', 'OPP_EFG_PCT_OFF',
       'OPP_EFG_PCT_DIFF', 'OPP_FTA_RATE_ON', 'OPP_FTA_RATE_OFF',
       'OPP_FTA_RATE_DIFF', 'OPP_TOV_PCT_ON', 'OPP_TOV_PCT_OFF',
       'OPP_TOV_PCT_DIFF', 'OPP_OREB_PCT_ON', 'OPP_OREB_PCT_OFF',
       'OPP_OREB_PCT_DIFF', 'MIN_ON', 'MIN_OFF', 'SEASON', 'USG_PCT',
       'PCT_AST', 'PCT_DREB', 'PCT_STL', 'PCT_BLK', 'PCT_BLKA', 'CREATION_PCT',
       'FINISHING_PCT', 'SHOOTING_PCT', 'TRANSITION_PCT', 'POSTUP_PCT',
       'MISC_PCT', 'ON_OFF_NET'],
      dtype='object')

In [24]:
df = model_df.copy()

# keep identifiers
id_cols = [
    "PLAYER_ID",
    "PLAYER_NAME",
    "TEAM_ID",
    "SEASON"
]


# minutes played on/off
df["MIN_TOTAL"] = (
    df["MIN_ON"] +
    df["MIN_OFF"]
)


# normalize how much evidence we have
df["ON_SHARE"] = (
    df["MIN_ON"] /
    df["MIN_TOTAL"]
)


df["OFF_SHARE"] = (
    df["MIN_OFF"] /
    df["MIN_TOTAL"]
)


print(df.shape)

(1260, 45)


In [25]:
model_df = df

In [26]:
weighted_cols = [
    "EFG_PCT_ON",
    "EFG_PCT_OFF",
    "EFG_PCT_DIFF",
    "FTA_RATE_ON",
    "FTA_RATE_OFF",
    "FTA_RATE_DIFF",
    "TM_TOV_PCT_ON",
    "TM_TOV_PCT_OFF",
    "TM_TOV_PCT_DIFF",
    "OREB_PCT_ON",
    "OREB_PCT_OFF",
    "OREB_PCT_DIFF",
    "OPP_EFG_PCT_ON",
    "OPP_EFG_PCT_OFF",
    "OPP_EFG_PCT_DIFF",
    "OPP_FTA_RATE_ON",
    "OPP_FTA_RATE_OFF",
    "OPP_FTA_RATE_DIFF",
    "OPP_TOV_PCT_ON",
    "OPP_TOV_PCT_OFF",
    "OPP_TOV_PCT_DIFF",
    "OPP_OREB_PCT_ON",
    "OPP_OREB_PCT_OFF",
    "OPP_OREB_PCT_DIFF",
    "USG_PCT",
    "PCT_AST",
    'CREATION_PCT', 
    'FINISHING_PCT', 
    'SHOOTING_PCT',
    'TRANSITION_PCT', 
    'POSTUP_PCT',
    'MISC_PCT',
    "PCT_DREB",
    "PCT_STL",
    "PCT_BLK",
    "PCT_BLKA",
    "ON_OFF_NET",
]

In [27]:
def merge_traded_players(df):
    rows = []

    for (player_id, season), group in df.groupby(["PLAYER_ID", "SEASON"]):
        w = group["MIN_ON"]

        row = {
            "PLAYER_ID": player_id,
            "SEASON": season,
            "MIN_ON": group["MIN_ON"].sum(),
            "MIN_OFF": group["MIN_OFF"].sum(),
        }

        row["MIN_TOTAL"] = row["MIN_ON"] + row["MIN_OFF"]
        row["ON_SHARE"] = row["MIN_ON"] / row["MIN_TOTAL"]

        for col in weighted_cols:
            row[col] = np.average(group[col], weights=w)

        rows.append(row)

    return pd.DataFrame(rows)

In [28]:
model_df = merge_traded_players(model_df)

In [29]:
model_df["USG_X_EFG"] = model_df["USG_PCT"] * model_df["EFG_PCT_DIFF"]
model_df["USG_X_TOV"] = model_df["USG_PCT"] * model_df["TM_TOV_PCT_DIFF"]
model_df["USG_X_FTA"] = model_df["USG_PCT"] * model_df["FTA_RATE_DIFF"]

In [30]:
model_df["CREATION_X_EFG"] = model_df["CREATION_PCT"] * model_df["EFG_PCT_DIFF"]
model_df["CREATION_X_TOV"] = model_df["CREATION_PCT"] * model_df["TM_TOV_PCT_DIFF"]
model_df["CREATION_X_FTA"] = model_df["CREATION_PCT"] * model_df["FTA_RATE_DIFF"]
model_df["CREATION_X_OREB"] = model_df["CREATION_PCT"] * model_df["OREB_PCT_DIFF"]

In [31]:
model_df["FINISHING_X_EFG"] = model_df["FINISHING_PCT"] * model_df["EFG_PCT_DIFF"]
model_df["FINISHING_X_TOV"] = model_df["FINISHING_PCT"] * model_df["TM_TOV_PCT_DIFF"]
model_df["FINISHING_X_FTA"] = model_df["FINISHING_PCT"] * model_df["FTA_RATE_DIFF"]
model_df["FINISHING_X_OREB"] = model_df["FINISHING_PCT"] * model_df["OREB_PCT_DIFF"]

In [32]:
model_df["SHOOTING_X_EFG"] = model_df["SHOOTING_PCT"] * model_df["EFG_PCT_DIFF"]
model_df["SHOOTING_X_TOV"] = model_df["SHOOTING_PCT"] * model_df["TM_TOV_PCT_DIFF"]
model_df["SHOOTING_X_FTA"] = model_df["SHOOTING_PCT"] * model_df["FTA_RATE_DIFF"]
model_df["SHOOTING_X_OREB"] = model_df["SHOOTING_PCT"] * model_df["OREB_PCT_DIFF"]

In [33]:
model_df["TRANSITION_X_EFG"] = model_df["TRANSITION_PCT"] * model_df["EFG_PCT_DIFF"]
model_df["TRANSITION_X_TOV"] = model_df["TRANSITION_PCT"] * model_df["TM_TOV_PCT_DIFF"]
model_df["TRANSITION_X_FTA"] = model_df["TRANSITION_PCT"] * model_df["FTA_RATE_DIFF"]
model_df["TRANSITION_X_OREB"] = model_df["TRANSITION_PCT"] * model_df["OREB_PCT_DIFF"]

In [34]:
model_df["POSTUP_X_EFG"] = model_df["POSTUP_PCT"] * model_df["EFG_PCT_DIFF"]
model_df["POSTUP_X_TOV"] = model_df["POSTUP_PCT"] * model_df["TM_TOV_PCT_DIFF"]
model_df["POSTUP_X_FTA"] = model_df["POSTUP_PCT"] * model_df["FTA_RATE_DIFF"]
model_df["POSTUP_X_OREB"] = model_df["POSTUP_PCT"] * model_df["OREB_PCT_DIFF"]

In [35]:
model_df["MISC_X_EFG"] = model_df["MISC_PCT"] * model_df["EFG_PCT_DIFF"]
model_df["MISC_X_TOV"] = model_df["MISC_PCT"] * model_df["TM_TOV_PCT_DIFF"]
model_df["MISC_X_FTA"] = model_df["MISC_PCT"] * model_df["FTA_RATE_DIFF"]
model_df["MISC_X_OREB"] = model_df["MISC_PCT"] * model_df["OREB_PCT_DIFF"]

In [36]:
K = 5000

model_df["OPP_EFG_SHRUNK"] = (
    model_df["OPP_EFG_PCT_DIFF"]
    * model_df["MIN_TOTAL"]
    / (model_df["MIN_TOTAL"] + K)
)

### Model Creation and Training

In [37]:
model_df = model_df.fillna(0)

In [38]:
# train test split
train_df = model_df[
    model_df["SEASON"].isin(
        ["2021-22", "2022-23", "2023-24", "2024-25"]
    )
].copy()

test_df = model_df[
    model_df["SEASON"] == "2025-26"
].copy()

In [39]:
# filtering out noisy samples
train_df = train_df[
    train_df["MIN_TOTAL"] > 1000
]

test_df = test_df[
    test_df["MIN_TOTAL"] > 1000
]

In [40]:
model_df.columns

Index(['PLAYER_ID', 'SEASON', 'MIN_ON', 'MIN_OFF', 'MIN_TOTAL', 'ON_SHARE',
       'EFG_PCT_ON', 'EFG_PCT_OFF', 'EFG_PCT_DIFF', 'FTA_RATE_ON',
       'FTA_RATE_OFF', 'FTA_RATE_DIFF', 'TM_TOV_PCT_ON', 'TM_TOV_PCT_OFF',
       'TM_TOV_PCT_DIFF', 'OREB_PCT_ON', 'OREB_PCT_OFF', 'OREB_PCT_DIFF',
       'OPP_EFG_PCT_ON', 'OPP_EFG_PCT_OFF', 'OPP_EFG_PCT_DIFF',
       'OPP_FTA_RATE_ON', 'OPP_FTA_RATE_OFF', 'OPP_FTA_RATE_DIFF',
       'OPP_TOV_PCT_ON', 'OPP_TOV_PCT_OFF', 'OPP_TOV_PCT_DIFF',
       'OPP_OREB_PCT_ON', 'OPP_OREB_PCT_OFF', 'OPP_OREB_PCT_DIFF', 'USG_PCT',
       'PCT_AST', 'CREATION_PCT', 'FINISHING_PCT', 'SHOOTING_PCT',
       'TRANSITION_PCT', 'POSTUP_PCT', 'MISC_PCT', 'PCT_DREB', 'PCT_STL',
       'PCT_BLK', 'PCT_BLKA', 'ON_OFF_NET', 'USG_X_EFG', 'USG_X_TOV',
       'USG_X_FTA', 'CREATION_X_EFG', 'CREATION_X_TOV', 'CREATION_X_FTA',
       'CREATION_X_OREB', 'FINISHING_X_EFG', 'FINISHING_X_TOV',
       'FINISHING_X_FTA', 'FINISHING_X_OREB', 'SHOOTING_X_EFG',
       'SHOOTING_X_TOV

In [43]:
features = [

    # offensive four factors
    # "EFG_PCT_ON",
    # "EFG_PCT_OFF",
    "EFG_PCT_DIFF",

    # "FTA_RATE_ON",
    # "FTA_RATE_OFF",
    "FTA_RATE_DIFF",

    # "TM_TOV_PCT_ON",
    # "TM_TOV_PCT_OFF",
    "TM_TOV_PCT_DIFF",

    # "OREB_PCT_ON",
    # "OREB_PCT_OFF",
    "OREB_PCT_DIFF",


    # defensive four factors
    # "OPP_EFG_PCT_ON",
    # "OPP_EFG_PCT_OFF",
    # "OPP_EFG_PCT_DIFF",
    # "OPP_EFG_SHRUNK",
    "PCT_DREB",
    "PCT_STL",
    "PCT_BLK",
    "PCT_BLKA",

    # "OPP_FTA_RATE_ON",
    # "OPP_FTA_RATE_OFF",
    "OPP_FTA_RATE_DIFF",

    # "OPP_TOV_PCT_ON",
    # "OPP_TOV_PCT_OFF",
    "OPP_TOV_PCT_DIFF",

    # "OPP_OREB_PCT_ON",
    # "OPP_OREB_PCT_OFF",
    "OPP_OREB_PCT_DIFF",

    # usage and playtype
    # "USG_PCT",
    # "PCT_AST",
    # "USG_X_EFG",
    # "USG_X_TOV",
    # "USG_X_FTA",
    # 'CREATION_PCT', 
    # 'FINISHING_PCT', 
    # 'SHOOTING_PCT',
    # 'TRANSITION_PCT', 
    # 'POSTUP_PCT',
    # 'MISC_PCT',
    'CREATION_X_EFG', 
    'CREATION_X_TOV', 
    'CREATION_X_FTA',
    'CREATION_X_OREB', 
    'FINISHING_X_EFG', 
    'FINISHING_X_TOV',
    'FINISHING_X_FTA', 
    'FINISHING_X_OREB', 
    'SHOOTING_X_EFG',
    'SHOOTING_X_TOV', 
    'SHOOTING_X_FTA', 
    'SHOOTING_X_OREB',
    'TRANSITION_X_EFG', 
    'TRANSITION_X_TOV', 
    'TRANSITION_X_FTA',
    'TRANSITION_X_OREB', 
    'POSTUP_X_EFG', 
    'POSTUP_X_TOV', 
    'POSTUP_X_FTA',
    'POSTUP_X_OREB', 
    'MISC_X_EFG', 
    'MISC_X_TOV', 
    'MISC_X_FTA',
    'MISC_X_OREB',

    # sample size controls
    # "MIN_ON",
    # "MIN_OFF",
    "MIN_TOTAL",
    "ON_SHARE"

]

In [44]:
X_train = train_df[features]
y_train = train_df["ON_OFF_NET"]

X_test = test_df[features]
y_test = test_df["ON_OFF_NET"]

In [45]:
season_order = {
    "2021-22":0,
    "2022-23":1,
    "2023-24":2,
    "2024-25":3,
    "2025-26":4
}

model_df["SEASON_NUM"] = (
    model_df["SEASON"]
    .map(season_order)
)

model_df = model_df.sort_values("SEASON_NUM")



In [46]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import numpy as np


tscv = TimeSeriesSplit(
    n_splits=3
)

X = X_train
y = y_train

alphas = [
    .01,
    .1,
    1,
    10,
    50,
    100
]


for alpha in alphas:

    model = Pipeline([
        (
            "scale",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=alpha)
        )
    ])


    scores = []


    for train_idx, val_idx in tscv.split(X):

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]


        model.fit(
            X_train,
            y_train
        )


        pred = model.predict(
            X_val
        )


        mae = mean_absolute_error(
            y_val,
            pred
        )

        scores.append(mae)


    print(
        alpha,
        np.mean(scores)
    )

0.01 3.384608520455991
0.1 3.382863046025945
1 3.368776092009876
10 3.3170101636894387
50 3.2527655999648455
100 3.229667475338477


In [47]:
model = Pipeline([
        (
            "scale",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=100)
        )
    ])

In [48]:
model.fit(X_train,y_train)

,steps,"[('scale', ...), ('ridge', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,100
,fit_intercept,True
,copy_X,True
,max_iter,None


In [63]:
predictions = model.predict(X_test)
results_df = test_df.assign(Predicted_Value=predictions)


In [64]:
seasons = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26"
]

name_map = {}
# team_map = {}

for season in seasons:
    df = LeagueDashPlayerStats(
        season=season,
        per_mode_detailed="PerGame"
    ).get_data_frames()[0]

    name_season_map = (
        df[["PLAYER_ID", "PLAYER_NAME"]]
        .drop_duplicates()
        .set_index("PLAYER_ID")["PLAYER_NAME"]
        .to_dict()
    )

    # team_season_map = (
    #     df[["TEAM_ID", "TEAM_ABBREVIATION"]]
    #     .drop_duplicates()
    #     .set_index("TEAM_ID")["TEAM_ABBREVIATION"]
    #     .to_dict()
    # )

    name_map.update(name_season_map)
    # team_map.update(team_season_map)

model_df["PLAYER_NAME"] = model_df["PLAYER_ID"].map(name_map)
# model_df["TEAM_ABBREVIATION"] = model_df["TEAM_ID"].map(team_map)


In [65]:
results_df["PLAYER_NAME"] = results_df["PLAYER_ID"].map(name_map)
# results_df["TEAM_ABBREVIATION"] = results_df["TEAM_ID"].map(team_map)

In [66]:
# check for missing players
missing = results_df[results_df["PLAYER_NAME"].isna()]
print(missing[["PLAYER_ID"]])

Empty DataFrame
Columns: [PLAYER_ID]
Index: []


### Results

In [75]:
# checking an individual player's features + result
print(results_df.sort_values(by='Predicted_Value', ascending=False).iloc[43].to_string())

PLAYER_ID                1629029
SEASON                   2025-26
MIN_ON                    1648.0
MIN_OFF                    433.0
MIN_TOTAL                 2081.0
ON_SHARE                0.791927
EFG_PCT_ON              0.586925
EFG_PCT_OFF             0.564822
EFG_PCT_DIFF            0.022103
FTA_RATE_ON             0.313982
FTA_RATE_OFF             0.31494
FTA_RATE_DIFF          -0.000958
TM_TOV_PCT_ON           0.134751
TM_TOV_PCT_OFF          0.146979
TM_TOV_PCT_DIFF        -0.012228
OREB_PCT_ON             0.258933
OREB_PCT_OFF                0.29
OREB_PCT_DIFF          -0.031067
OPP_EFG_PCT_ON          0.561099
OPP_EFG_PCT_OFF         0.547053
OPP_EFG_PCT_DIFF        0.014046
OPP_FTA_RATE_ON         0.210115
OPP_FTA_RATE_OFF        0.235741
OPP_FTA_RATE_DIFF      -0.025627
OPP_TOV_PCT_ON          0.136483
OPP_TOV_PCT_OFF         0.158942
OPP_TOV_PCT_DIFF       -0.022459
OPP_OREB_PCT_ON         0.287817
OPP_OREB_PCT_OFF        0.308501
OPP_OREB_PCT_DIFF      -0.020684
USG_PCT   

In [54]:
mae = mean_absolute_error(
            y_test,
            predictions
        )

In [55]:
mae

3.3279999455518583

In [57]:
results_df = results_df[['PLAYER_NAME', 'Predicted_Value']]
df_sorted = results_df.sort_values(by='Predicted_Value', ascending=False)
df_sorted

,PLAYER_NAME,Predicted_Value
286,Nikola Jokić,15.486406
213,Giannis Antetokounmpo,13.493412
1130,Victor Wembanyama,11.346489
427,Pascal Siakam,11.240269
401,Jamal Murray,10.389079
...,...,...
935,Trey Murphy III,-5.338680
875,Isaiah Stewart,-7.900457
1177,Ronald Holland II,-8.450888
922,Vít Krejčí,-8.927405


In [58]:
# exporting results
df_sorted.to_csv("outputs/four_factor_model_results_v8.csv", index=False)

### Feature Importances

In [59]:
regression_pipe_coefs = pd.DataFrame(
    data=model.named_steps["ridge"].coef_, index=features, columns=["Coefficient"]
)

In [60]:
regression_pipe_coefs.sort_values(by="Coefficient", ascending=False).head(80)

,Coefficient
OPP_TOV_PCT_DIFF,0.886574
EFG_PCT_DIFF,0.881736
ON_SHARE,0.759153
OREB_PCT_DIFF,0.663780
POSTUP_X_EFG,0.567346
MISC_X_EFG,0.560893
PCT_BLK,0.539619
TRANSITION_X_OREB,0.425130
FTA_RATE_DIFF,0.421962
PCT_STL,0.407805
